In [1]:
import arxiv 
import sys 
from typing import List 
import pprint
from pathlib import Path 
import os 
import anthropic
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv()

## Metadata

In [2]:
PAPER_DIR = "papers"
INFO_FILE_NAME = 'papers.json'

## Define llm

In [3]:
llm = anthropic.Anthropic()

## Add function for searching the paper 

In [4]:
def search_papers(topic: str, max_results: int = 10) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information. 

    Args: 
        topic: The topic to search for
        max_results: Maximum number of results to retrieve 

    Returns: 
        List of paper IDs found in the search
    """
    # Define client 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    new_papers_info = {
        p.get_short_id(): {
            'title': p.title, 
            'authors': [author.name for author in p.authors], 
            'summary': p.summary, 
            'published': str(p.published.date())
        }

        for p in papers
    }
    
    # Save this info along with other papers    
    dir_name = Path(PAPER_DIR)/ topic.lower().replace(" ", "_")
    file_name = Path(dir_name) / INFO_FILE_NAME
    os.makedirs(dir_name, exist_ok=True)

    # Read if there is an existing search on this topic 
    try: 
        with open(file_name, 'r') as json_file: 
            existing_papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError): 
        existing_papers_info = {}

    # Combine existing and new papers
    combined_papers_info = {**existing_papers_info, **new_papers_info}

    # Write into JSON file
    with open(file_name, 'w') as json_file: 
        json.dump(combined_papers_info, json_file, indent=2)

    print(f"Results updated in the file:\n{str(file_name)}")

    return list(new_papers_info.keys())



In [5]:
search_papers("Manny")

Results updated in the file:
papers/manny/papers.json


['1908.02121v1',
 '1908.06877v1',
 '1907.13214v1',
 '2208.06882v2',
 '1510.01942v1',
 '2601.12020v1',
 '1103.5433v6',
 'cmp-lg/9505045v1',
 'cmp-lg/9405024v1',
 'cmp-lg/9407010v1']

## Add function for extracting the paper info for a given paper ID

In [6]:
def extract_info(paper_id: str) -> dict:
    """
    Search all topic folders for information about a specific paper.

    Args:
        paper_id: The arXiv ID of the paper to look for (e.g. '1908.02121v1')

    Returns:
        Dictionary with the paper's title, authors, summary and published date,
        or an empty dictionary if the paper isn't found in any saved search
    """
    for p in Path(PAPER_DIR).iterdir():
        if p.is_dir():
            file_path = p / INFO_FILE_NAME
            try: 
                with open(file_path, 'r') as json_file: 
                    file_contents = json.load(json_file)
                return json.dumps(file_contents[paper_id], indent=2)
            except (FileNotFoundError, json.JSONDecodeError, KeyError): 
                pass
    return {}

pprint.pp(extract_info('1907.13214v1'))

('{\n'
 '  "title": "Alexa as a CALL platform for children: Where do we start?",\n'
 '  "authors": [\n'
 '    "Nikos Tsourakis",\n'
 '    "Manny Rayner",\n'
 '    "Hanieh Habibi",\n'
 '    "Pierre-Emmanuel Gallais",\n'
 '    "Cathy Chua",\n'
 '    "Matt Butterweck"\n'
 '  ],\n'
 '  "summary": "Amazon\'s Alexa is now widely available and shows interesting '
 'potential as a platform for hosting CALL games aimed at children. In this '
 'paper, we describe an initial informal experiment where we created some '
 'simple CALL games and made them available to a few child testers. We report '
 "the children's and parents' reactions. Our overall conclusion is that, "
 'although Alexa has many positive features, there are still fundamental '
 'platform issues in the current version that make it very difficult to build '
 'compelling CALL games for children. The games used will soon be freely '
 'available for download on the Alexa store.",\n'
 '  "published": "2019-07-30"\n'
 '}')


## Process Query

In [19]:
def process_query(query: str):
    _MODEL_NAME = 'claude-sonnet-4-6'
    messages = [
        {
            'role': 'user', 
            'content': query
        }
    ]

    _AVAILABLE_TOOLS = [
        anthropic.beta_tool(f).to_dict()
        for f in [search_papers, extract_info]
    ]

    response = llm.messages.create(
        max_tokens=2024, 
        model=_MODEL_NAME,
        messages=messages, 
        tools=_AVAILABLE_TOOLS
    )

    print(response)

    process_query = True
    while process_query: 
        has_tool_usage = False 
        assistant_messages = []
        tool_use_messages = []

        for content in response.content:
            if content.type == 'text': 
                print(content.text)
                assistant_messages.append(content)
            elif content.type == 'tool_use':
                has_tool_usage = True
                assistant_messages.append(content) 

                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name}")

                tool_output = globals()[tool_name](**tool_args)
                if tool_output is None: 
                    tool_output = "The operation completed but didn't return any results."
                elif isinstance(tool_output, list): 
                    tool_output = ', '.join(tool_output)
                elif isinstance(tool_output, dict): 
                    tool_output = json.dumps(tool_output, indent=2)
                else: 
                    tool_output = str(tool_output)

                tool_use_messages.append(
                    {
                        'type': 'tool_result', 
                        'tool_use_id': tool_id, 
                        'content': tool_output
                    }
                )
            else: 
                print(content.type)
                raise NotImplementedError("Not yet implemented")
                
        messages.append({'role': 'assistant', 'content': assistant_messages})
        if has_tool_usage:
            messages.append({'role': 'user', 'content': tool_use_messages})
            response = llm.messages.create(
                max_tokens=2024, 
                model=_MODEL_NAME,
                messages=messages,
                tools=_AVAILABLE_TOOLS
            )
        else: 
            process_query = False
    

## Chat loop 

In [20]:
def run_chat_loop():
    print("Type your query below or enter 'quit' to quit")
    while True: 
        try: 
            query = input("Please enter your query: ")
            if query.lower().strip() == 'quit':
                break 
            process_query(query)
            print("Iteration completed")
        except Exception as e: 
            print(f"Error processing query: {str(e)}")

run_chat_loop()

Type your query below or enter 'quit' to quit


Please enter your query:  Find arxiv papers written by chalana


Message(id='msg_011CgQUBDwUrqN3GjxNZTxJn', container=None, content=[TextBlock(citations=None, text='I\'ll search for papers by "chalana" across multiple relevant topics simultaneously to give you comprehensive results!', type='text'), ToolUseBlock(id='toolu_01Lpc4E6ugbit9ky8FDiwA3h', caller=DirectCaller(type='direct'), input={'topic': 'chalana'}, name='search_papers', type='tool_use', toolset_name=None), ToolUseBlock(id='toolu_018Cyh2DBM5syJ1W2gqPypjw', caller=DirectCaller(type='direct'), input={'topic': 'author:chalana'}, name='search_papers', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=741, output_tokens=115, output_tokens_details=None, server_tool_use=None, 

Please enter your query:  Please find papers about ai 


Message(id='msg_011CgQULMNhdfchyoaL5dvnE', container=None, content=[ToolUseBlock(id='toolu_01DYandcBnd9xab2Zc8hk1tn', caller=DirectCaller(type='direct'), input={'topic': 'ai'}, name='search_papers', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=739, output_tokens=54, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'))
Calling tool search_papers
Results updated in the file:
papers/ai/papers.json
Now let me extract the information for all these papers simultaneously!
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Cal

Please enter your query:  Please find top 15 papers about ai 


Message(id='msg_011CgQUS44pZJSzz2tPLFcu6', container=None, content=[ToolUseBlock(id='toolu_0113863Vq985u3YdHe7eJrQJ', caller=DirectCaller(type='direct'), input={'topic': 'ai', 'max_results': 15}, name='search_papers', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=743, output_tokens=73, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'))
Calling tool search_papers
Results updated in the file:
papers/ai/papers.json
Now let me extract information for all 15 papers simultaneously!
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extract_info
Calling tool extr

KeyboardInterrupt: Interrupted by user